# **Proyek AI/ML & MLflow Bank Marketing (Databricks Free Edition)**
## **Tahap 4: Klasifikasi Prediksi Deposito (`y_flag`), Pelacakan MLflow (Unity Catalog) & Pengujian Batch**

### **Keputusan Desain Pemodelan:**
1. **Jenis Tugas:** Klasifikasi Biner dengan target diskret `y_flag` (`1` = nasabah berlangganan deposito `yes`, `0` = tidak berlangganan `no`).
2. **Penanganan *Target Leakage* (Mengeluarkan `duration`):**
   * Sesuai dokumentasi dataset UCI (`bank-names.txt`), `duration` (durasi telepon) baru diketahui **setelah** panggilan selesai dan sangat menentukan nilai `y`.
   * Untuk model prediktif pra-panggilan yang realistis di produksi, **`duration` sengaja dikeluarkan dari fitur training** sehingga tidak terjadi kebocoran data (*target leakage*).
3. **Penanganan Kelas Tidak Seimbang (*Imbalanced Classes*):**
   * Proporsi kelas positif (`yes`) hanya ~11,7% dibanding kelas negatif (`no`) ~88,3%.
   * Digunakan **Stratified Train/Test Split (80/20)** dan bobot kelas seimbang (`class_weight="balanced"`) agar model tetap peka mendeteksi calon nasabah potensial (mengoptimalkan Recall & F1-Score).
4. **Kompatibilitas Serverless & Model Serving pada Free Edition:**
   * Model dilatih menggunakan `scikit-learn` `Pipeline`, dilacak dengan **MLflow**, dan didaftarkan ke **Unity Catalog Model Registry** dengan alias `@prod`.
   * Karena **Databricks Free Edition** tidak menyediakan Model Serving endpoint *real-time*, pengujian dilakukan melalui pemuatan model ter-register `@prod` dan **uji prediksi sampel & batch inference** ke tabel Delta `gold_ml_predictions`.

In [ ]:
%pip install -q mlflow scikit-learn

---
### **1. Parameter & Konfigurasi Unity Catalog + MLflow**

In [ ]:
import json
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F
from pyspark.sql import types as T
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Matikan autolog untuk mencegah kendala serialisasi PlanMetrics pada Spark Connect/Serverless
mlflow.autolog(disable=True)
mlflow.sklearn.autolog(disable=True)
mlflow.set_registry_uri("databricks-uc")

dbutils.widgets.text("catalog", "workspace", "Unity Catalog Name")
dbutils.widgets.text("schema", "bank_marketing", "Schema Name")
dbutils.widgets.text("silver_table", "silver_bank_marketing", "Nama Tabel Delta Silver")
dbutils.widgets.text("model_name", "bank_marketing_deposit_clf", "Nama Model di Unity Catalog")
dbutils.widgets.text("predictions_table", "gold_ml_predictions", "Nama Tabel Delta Prediksi Batch")
dbutils.widgets.text("test_size", "0.2", "Proporsi Data Uji")
dbutils.widgets.text("random_state", "42", "Random Seed")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
silver_table = dbutils.widgets.get("silver_table").strip()
model_name = dbutils.widgets.get("model_name").strip()
predictions_table = dbutils.widgets.get("predictions_table").strip()
test_size = float(dbutils.widgets.get("test_size").strip())
random_state = int(dbutils.widgets.get("random_state").strip())

full_silver_table = f"{catalog}.{schema}.{silver_table}"
full_model_name = f"{catalog}.{schema}.{model_name}"
full_predictions_table = f"{catalog}.{schema}.{predictions_table}"

if not spark.catalog.tableExists(full_silver_table):
    raise ValueError(f"Tabel Silver {full_silver_table} tidak ditemukan. Jalankan pipeline ETL terlebih dahulu.")

print(f"Silver Table     : {full_silver_table}")
print(f"UC Model Name    : {full_model_name}")
print(f"Predictions Table: {full_predictions_table}")

---
### **2. Muat Data Silver, Cek Distribusi Kelas & Split Train/Test**

In [ ]:
NUMERIC_FEATURES = [
    "age", "balance", "day", "month_num",
    "campaign", "pdays", "previous", "was_previously_contacted",
]
CATEGORICAL_FEATURES = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "poutcome",
]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_COL = "y_flag"

select_cols = ALL_FEATURES + [TARGET_COL, "campaign_year", "year_month", "age_group", "balance_segment"]
pdf = spark.table(full_silver_table).select(*select_cols).toPandas()

# Casting numerik ke float64 agar aman terhadap infer_signature MLflow
for col_name in NUMERIC_FEATURES:
    pdf[col_name] = pdf[col_name].astype("float64")
pdf[TARGET_COL] = pdf[TARGET_COL].astype("int64")

class_counts = pdf[TARGET_COL].value_counts().sort_index()
total_n = len(pdf)
print("=== DISTRIBUSI KELAS TARGET (y_flag) ===")
for cls_val, cnt in class_counts.items():
    label_txt = "yes (1)" if cls_val == 1 else "no (0)"
    print(f"Kelas {label_txt}: {cnt:,} baris ({cnt * 100.0 / total_n:.2f}%)")

X = pdf[ALL_FEATURES]
y = pdf[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=random_state
)
print(f"\nUkuran Data Latih (Train): {len(X_train):,} baris (positif: {int(y_train.sum()):,})")
print(f"Ukuran Data Uji (Test)   : {len(X_test):,} baris (positif: {int(y_test.sum()):,})")

---
### **3. Pelatihan Baseline & Model Pembanding + Pelacakan MLflow**

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ]
)

candidate_models = [
    (
        "Baseline_LogisticRegression",
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_state),
        {"model_type": "LogisticRegression", "max_iter": 1000, "class_weight": "balanced"},
    ),
    (
        "Challenger_RandomForest",
        RandomForestClassifier(
            n_estimators=150,
            max_depth=12,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        {"model_type": "RandomForestClassifier", "n_estimators": 150, "max_depth": 12, "class_weight": "balanced"},
    ),
    (
        "Challenger_HistGradientBoosting",
        HistGradientBoostingClassifier(
            max_iter=150,
            max_depth=6,
            learning_rate=0.08,
            class_weight="balanced",
            random_state=random_state,
        ),
        {"model_type": "HistGradientBoostingClassifier", "max_iter": 150, "max_depth": 6, "learning_rate": 0.08, "class_weight": "balanced"},
    ),
]

X_sample = X_train.head(100)
experiment_results = []

for run_name, estimator, params_dict in candidate_models:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", estimator),
    ])
    pipe.fit(X_train, y_train)

    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)
    y_train_prob = pipe.predict_proba(X_train)[:, 1]
    y_test_prob = pipe.predict_proba(X_test)[:, 1]

    metrics_dict = {
        "train_accuracy": float(accuracy_score(y_train, y_train_pred)),
        "train_precision": float(precision_score(y_train, y_train_pred, zero_division=0)),
        "train_recall": float(recall_score(y_train, y_train_pred, zero_division=0)),
        "train_f1": float(f1_score(y_train, y_train_pred, zero_division=0)),
        "train_roc_auc": float(roc_auc_score(y_train, y_train_prob)),
        "test_accuracy": float(accuracy_score(y_test, y_test_pred)),
        "test_precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
        "test_recall": float(recall_score(y_test, y_test_pred, zero_division=0)),
        "test_f1": float(f1_score(y_test, y_test_pred, zero_division=0)),
        "test_roc_auc": float(roc_auc_score(y_test, y_test_prob)),
    }

    cm = confusion_matrix(y_test, y_test_pred)
    tn, fp, fn, tp = [int(v) for v in cm.ravel()]

    signature = infer_signature(X_sample, pipe.predict(X_sample))

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({**params_dict, "test_size": test_size, "random_state": random_state, "exclude_duration": True})
        mlflow.log_metrics({**metrics_dict, "test_tn": tn, "test_fp": fp, "test_fn": fn, "test_tp": tp})
        mlflow.sklearn.log_model(
            pipe,
            artifact_path="model",
            signature=signature,
            input_example=X_sample.head(5),
        )
        run_id = run.info.run_id

    print(f"\n=== {run_name} (run_id={run_id}) ===")
    print(f"Confusion Matrix Test -> TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(classification_report(y_test, y_test_pred, target_names=["no (0)", "yes (1)"], digits=4))

    experiment_results.append({
        "run_name": run_name,
        "run_id": run_id,
        **metrics_dict,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    })

results_df = pd.DataFrame(experiment_results).sort_values("test_f1", ascending=False)
print("=== PERBANDINGAN METRIK SELURUH MODEL ===")
display(results_df[["run_name", "train_f1", "test_precision", "test_recall", "test_f1", "test_roc_auc", "test_accuracy"]])

---
### **4. Registrasi Model Terbaik ke Unity Catalog & Penetapan Alias `@prod`**

In [ ]:
# Pilih model terbaik berdasarkan test_f1 menggunakan sort + indeks (hindari shadow builtin max)
best_model_info = sorted(experiment_results, key=lambda r: r["test_f1"], reverse=True)[0]
best_run_id = best_model_info["run_id"]
best_run_name = best_model_info["run_name"]

reg_version = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name=full_model_name,
)

uc_client = MlflowClient(registry_uri="databricks-uc")
uc_client.set_registered_model_alias(
    name=full_model_name,
    alias="prod",
    version=reg_version.version,
)

print("=== REGISTRASI MODEL UNITY CATALOG BERHASIL ===")
print(f"Model Terpilih : {best_run_name}")
print(f"Test F1-Score  : {best_model_info['test_f1']:.4f}")
print(f"Test ROC-AUC   : {best_model_info['test_roc_auc']:.4f}")
print(f"UC Model URI   : models:/{full_model_name}@prod (versi {reg_version.version})")

---
### **5. Pengujian Model (`@prod`): Prediksi Sampel Manual & Batch Inference ke Tabel Delta**

Karena **Databricks Free Edition** tidak mendukung *real-time Model Serving endpoint*, pengujian dilakukan langsung menggunakan artefak model resmi Unity Catalog (`models:/<catalog>.<schema>.<model_name>@prod`) untuk:
1. Menguji **3 profil input nasabah sampel** dan mencatat input, output, serta status prediksi.
2. Menguji **prediksi batch** pada seluruh data uji (*test set*) dan menyimpan hasilnya ke tabel Delta `{catalog}.{schema}.{predictions_table}`.

In [ ]:
prod_model_uri = f"models:/{full_model_name}@prod"
loaded_prod_model = mlflow.pyfunc.load_model(prod_model_uri)

# 5A. Pengujian Contoh Input Prediksi (3 Profil Nasabah Sampel)
sample_inputs = pd.DataFrame([
    {
        "age": 32.0, "balance": 3500.0, "day": 15.0, "month_num": 5.0,
        "campaign": 1.0, "pdays": 180.0, "previous": 2.0, "was_previously_contacted": 1.0,
        "job": "management", "marital": "single", "education": "tertiary",
        "default": "no", "housing": "no", "loan": "no", "contact": "cellular", "poutcome": "success",
    },
    {
        "age": 46.0, "balance": 120.0, "day": 20.0, "month_num": 5.0,
        "campaign": 6.0, "pdays": -1.0, "previous": 0.0, "was_previously_contacted": 0.0,
        "job": "blue-collar", "marital": "married", "education": "secondary",
        "default": "no", "housing": "yes", "loan": "yes", "contact": "unknown", "poutcome": "unknown",
    },
    {
        "age": 61.0, "balance": 4800.0, "day": 12.0, "month_num": 10.0,
        "campaign": 1.0, "pdays": -1.0, "previous": 0.0, "was_previously_contacted": 0.0,
        "job": "retired", "marital": "married", "education": "tertiary",
        "default": "no", "housing": "no", "loan": "no", "contact": "cellular", "poutcome": "unknown",
    },
])

sample_preds = loaded_prod_model.predict(sample_inputs[ALL_FEATURES]).astype("int64")
sample_results = sample_inputs.copy()
sample_results["prediction"] = sample_preds
sample_results["predicted_label"] = sample_results["prediction"].map({1: "yes", 0: "no"})
sample_results["inference_status"] = "SUCCESS"

print("=== HASIL UJI PREDIKSI 3 SAMPEL PROFIL NASABAH ===")
display(sample_results[["age", "job", "balance", "poutcome", "contact", "prediction", "predicted_label", "inference_status"]])

# 5B. Pengujian Batch pada Data Uji (Driver-Side Inference, Serverless-Compatible)
batch_test_df = pdf.loc[X_test.index].copy()
batch_test_df["prediction"] = loaded_prod_model.predict(X_test[ALL_FEATURES]).astype("int64")
batch_test_df["predicted_label"] = batch_test_df["prediction"].map({1: "yes", 0: "no"})
batch_test_df["actual_label"] = batch_test_df[TARGET_COL].map({1: "yes", 0: "no"})

batch_spark_df = spark.createDataFrame(batch_test_df)
batch_spark_df = (
    batch_spark_df
    .withColumn("prediction", F.col("prediction").cast(T.IntegerType()))
    .withColumn("y_flag", F.col("y_flag").cast(T.IntegerType()))
    .withColumn("model_version", F.lit(str(reg_version.version)))
    .withColumn("scored_at", F.current_timestamp())
)

(
    batch_spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_predictions_table)
)

batch_cm = confusion_matrix(batch_test_df[TARGET_COL], batch_test_df["prediction"])
batch_f1 = float(f1_score(batch_test_df[TARGET_COL], batch_test_df["prediction"], zero_division=0))
batch_acc = float(accuracy_score(batch_test_df[TARGET_COL], batch_test_df["prediction"]))

print(f"\n=== HASIL UJI BATCH ({len(batch_test_df):,} baris test set) ===")
print(f"Disimpan ke Tabel Delta : {full_predictions_table}")
print(f"Batch Accuracy          : {batch_acc:.4f}")
print(f"Batch F1-Score          : {batch_f1:.4f}")
print(f"Batch Confusion Matrix  : {batch_cm.tolist()}")

exit_payload = {
    "status": "SUCCESS",
    "best_model": best_run_name,
    "registered_model": full_model_name,
    "registered_version": str(reg_version.version),
    "alias": "prod",
    "test_f1": round(best_model_info["test_f1"], 4),
    "test_roc_auc": round(best_model_info["test_roc_auc"], 4),
    "predictions_table": full_predictions_table,
    "batch_rows_scored": int(len(batch_test_df)),
}
dbutils.notebook.exit(json.dumps(exit_payload))